# Demo A: From a Hugging Face Checkpoint to Tokens on the Wire

**ODSC 2026** | LLM Inference at Scale | [Harshul Jain](https://github.com/harshuljain13)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/demo_a_model_memory_prefill_decode.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/demo_a_model_memory_prefill_decode.ipynb)

**The story this notebook tells, end to end:**

| # | Section | The question it answers |
|---|---------|-------------------------|
| 1 | Download the model | What did we actually pull from Hugging Face? |
| 2 | Where the memory goes | Which bytes on the GPU belong to what? |
| 3 | First inference | Why does a long prompt feel slower than a short one? |
| 4 | Splitting the clock | Where inside that latency did the time go? |
| 5 | The bandwidth floor | Why can't decode go faster? |
| 6 | The KV cache | What is holding decode up at all? |

**Every number below is measured on the machine running this notebook.** No formulas
standing in for benchmarks, no simulated timings. Where we *predict* a value from the
model config, it is labelled as a prediction and then checked against a measurement.

**Requirements:** a CUDA GPU with >= 18 GB VRAM runs the 7B path. Anything
smaller (or CPU-only) automatically falls back to a 1.1B model and smaller knobs —
the story is identical, the absolute numbers are not.

## 0. Setup

Install, import, and report exactly what hardware we are on. Every measurement later
is relative to this.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'pandas', 'accelerate'])

import gc
import time

import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')

if HAS_CUDA:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Device : {gpu_name}')
    print(f'VRAM   : {vram_gb:.1f} GB')
else:
    gpu_name, vram_gb = 'CPU', 0.0
    print('Device : CPU (no CUDA found)')
    print('Timings will be slow but the prefill/decode asymmetry still shows up.')

print(f'torch  : {torch.__version__}')

In [ ]:
# --- Configuration -------------------------------------------------------
# The 7B path needs ~18 GB. Below that we drop to TinyLlama so the notebook
# still runs end to end instead of OOM-ing halfway through the story.
BIG_MODEL = 'mistralai/Mistral-7B-v0.1'          # non-gated, no HF token needed
SMALL_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

USE_BIG = HAS_CUDA and vram_gb >= 18
MODEL_ID = BIG_MODEL if USE_BIG else SMALL_MODEL
DTYPE = torch.float16 if HAS_CUDA else torch.float32

# Knobs, scaled to the hardware so a laptop run finishes in minutes.
SHORT_PROMPT = 'Explain quantum computing in simple terms:'
FILLER = 'The quick brown fox jumps over the lazy dog. '
LONG_TOKENS = 4096 if USE_BIG else (1024 if HAS_CUDA else 256)
DECODE_STEPS = 32 if HAS_CUDA else 8
NO_CACHE_STEPS = 12 if HAS_CUDA else 4
SWEEP_LENGTHS = [128, 512, 1024, 2048, 4096] if USE_BIG else (
    [128, 256, 512, 1024] if HAS_CUDA else [64, 128, 256])

print(f'Model        : {MODEL_ID}')
print(f'Precision    : {str(DTYPE).replace("torch.", "")}')
print(f'Long prompt  : {LONG_TOKENS} tokens')
print(f'Decode steps : {DECODE_STEPS}')
print(f'Prefill sweep: {SWEEP_LENGTHS}')

## 1. What Did We Just Download?

A Hugging Face model id is a pointer to a pile of tensor files. Before we talk about
speed, let's see what arrived: how long the download and load took, how much disk it
occupies, and how many parameters are really in there.

In [ ]:
# Load the config first — it is tiny, and it tells us the shape of what is coming.
config = AutoConfig.from_pretrained(MODEL_ID)

n_layers = config.num_hidden_layers
d_model = config.hidden_size
n_heads = config.num_attention_heads
n_kv_heads = getattr(config, 'num_key_value_heads', None) or n_heads
head_dim = getattr(config, 'head_dim', None) or (d_model // n_heads)

print('--- Architecture (from config.json) ---')
print(f'Layers (L)          : {n_layers}')
print(f'Hidden dim (d)      : {d_model}')
print(f'Attention heads     : {n_heads}')
print(f'KV heads            : {n_kv_heads}' +
      ('  <-- GQA: fewer KV heads than Q heads' if n_kv_heads < n_heads else '  (no GQA)'))
print(f'Head dim            : {head_dim}')
print(f'Max context         : {getattr(config, "max_position_embeddings", "?")}')

In [ ]:
# Download + load, timed. The first run pays for the network; re-runs hit the HF cache.
print(f'Loading {MODEL_ID} ... (first run downloads several GB)')

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map='auto' if HAS_CUDA else None)
model.eval()
if not HAS_CUDA:
    model.to(DEVICE)
load_sec = time.perf_counter() - t0

n_params = sum(p.numel() for p in model.parameters())
param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())

print(f'\nLoad time           : {load_sec:.1f} s')
print(f'Parameters          : {n_params/1e9:.2f} B')
print(f'Bytes per parameter : {param_bytes/n_params:.2f}')
print(f'Weights on device   : {param_bytes/1e9:.2f} GB')

In [ ]:
# How much disk did this checkpoint actually take? (Reads the local HF cache.)
try:
    from huggingface_hub import scan_cache_dir

    cache = scan_cache_dir()
    repo = next((r for r in cache.repos if r.repo_id == MODEL_ID), None)
    if repo is not None:
        print(f'On-disk cache       : {repo.size_on_disk/1e9:.2f} GB')
        print(f'Files               : {sum(len(rev.files) for rev in repo.revisions)}')
    else:
        print(f'{MODEL_ID} not found in the local cache scan.')
except Exception as exc:
    print(f'Cache scan unavailable: {exc}')

## 2. Where Does the Memory Go?

"A 7B model needs 14 GB" is the headline, but it hides the shape of the spend. Let's
bucket *every parameter tensor by name* and add them up — this is a true accounting of
the loaded model, not an estimate from the config.

In [ ]:
# Classify every parameter tensor into a component bucket. Priority matters:
# 'lm_head' must be checked before the generic attention/MLP patterns.
def classify(name):
    n = name.lower()
    if 'lm_head' in n:
        return 'LM head'
    if any(k in n for k in ('embed_tokens', 'wte', 'word_embeddings', 'embeddings')):
        return 'Embeddings'
    if 'norm' in n or 'ln_' in n:
        return 'Norms'
    if any(k in n for k in ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'attn', 'attention')):
        return 'Attention (Q,K,V,O)'
    if any(k in n for k in ('mlp', 'gate_proj', 'up_proj', 'down_proj', 'fc1', 'fc2', 'ffn')):
        return 'MLP / FFN'
    return 'Other'


buckets = {}
for name, p in model.named_parameters():
    key = classify(name)
    nb = p.numel() * p.element_size()
    agg = buckets.setdefault(key, {'bytes': 0, 'params': 0, 'tensors': 0})
    agg['bytes'] += nb
    agg['params'] += p.numel()
    agg['tensors'] += 1

mem_table = pd.DataFrame([
    {'Component': k,
     'Params (M)': round(v['params'] / 1e6, 1),
     'Memory (GB)': round(v['bytes'] / 1e9, 3),
     'Share of weights': f"{v['bytes']/param_bytes*100:.1f}%",
     'Tensors': v['tensors']}
    for k, v in sorted(buckets.items(), key=lambda kv: -kv[1]['bytes'])
])
mem_table.loc[len(mem_table)] = ['TOTAL', round(n_params/1e6, 1),
                                 round(param_bytes/1e9, 3), '100.0%',
                                 int(mem_table['Tensors'].sum())]
mem_table

In [ ]:
# The allocator's view vs. the sum of parameter tensors. The gap is real overhead:
# buffers, alignment, and anything the loader kept around.
if HAS_CUDA:
    torch.cuda.synchronize()
    allocated_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9
    free_b, total_b = torch.cuda.mem_get_info()

    print(f'Sum of weight tensors : {param_bytes/1e9:.2f} GB')
    print(f'torch allocated       : {allocated_gb:.2f} GB')
    print(f'torch reserved        : {reserved_gb:.2f} GB')
    print(f'Overhead vs weights   : {allocated_gb - param_bytes/1e9:.2f} GB')
    print()
    print(f'GPU free right now    : {free_b/1e9:.2f} GB of {total_b/1e9:.2f} GB')
    print(f'--> Everything else — KV cache for every concurrent user, activations,')
    print(f'    fragmentation — has to fit in that {free_b/1e9:.1f} GB.')
else:
    allocated_gb = param_bytes / 1e9
    free_b = total_b = 0
    print(f'Weights in host RAM: {param_bytes/1e9:.2f} GB (CPU run — no VRAM budget to show)')

In [ ]:
# Two views: what the weights are made of, and what is left over for users.
fig, (ax_comp, ax_budget) = plt.subplots(1, 2, figsize=(13, 4.5),
                                         gridspec_kw={'width_ratios': [1.6, 1]})

comps = mem_table[mem_table['Component'] != 'TOTAL']
palette = ['#dbeafe', '#dcfce7', '#fef3c7', '#ffedd5', '#ffe4e6', '#f3e8ff']
ax_comp.barh(comps['Component'], comps['Memory (GB)'],
             color=palette[:len(comps)], edgecolor='#000', linewidth=1.2)
ax_comp.invert_yaxis()
ax_comp.set_xlabel('Memory (GB)', fontsize=11)
ax_comp.set_title(f'What the {n_params/1e9:.1f}B weights are made of', fontsize=12, fontweight='bold')
for i, (val, share) in enumerate(zip(comps['Memory (GB)'], comps['Share of weights'])):
    ax_comp.text(val + comps['Memory (GB)'].max() * 0.02, i, f'{val:.2f} GB  ({share})',
                 va='center', fontsize=9)
ax_comp.set_xlim(0, comps['Memory (GB)'].max() * 1.35)
for side in ('top', 'right'):
    ax_comp.spines[side].set_visible(False)

if HAS_CUDA:
    used_gb = (total_b - free_b) / 1e9
    ax_budget.bar(['GPU'], [used_gb], color='#ffe4e6', edgecolor='#000',
                  linewidth=1.2, label=f'In use: {used_gb:.1f} GB')
    ax_budget.bar(['GPU'], [free_b / 1e9], bottom=[used_gb], color='#f3f4f6',
                  edgecolor='#000', linewidth=1.2, label=f'Free: {free_b/1e9:.1f} GB')
    ax_budget.set_ylabel('VRAM (GB)', fontsize=11)
    ax_budget.set_title('GPU budget after loading\n(nothing served yet)', fontsize=12, fontweight='bold')
    ax_budget.legend(fontsize=9, loc='upper right')
else:
    ax_budget.text(0.5, 0.5, 'CPU run —\nno VRAM budget', ha='center', va='center', fontsize=12)
    ax_budget.set_xticks([])
    ax_budget.set_yticks([])
for side in ('top', 'right'):
    ax_budget.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

### The One Cost That Is Not Fixed

Weights are a one-time cost: load them once, serve everyone. The **KV cache** is
per-user *and* per-token, and it is the reason a GPU that comfortably holds the model
still falls over under concurrency.

From the config we can *predict* what one cached token should cost:

$$\text{bytes per token} = 2 \times L \times n_{kv} \times d_h \times \text{bytes per element}$$

The leading 2 is one K vector plus one V vector. We will check this prediction against
a real cache in Section 6 — don't trust it yet.

In [ ]:
elem_bytes = torch.finfo(DTYPE).bits // 8
predicted_bytes_per_token = 2 * n_layers * n_kv_heads * head_dim * elem_bytes

print(f'Predicted KV per token : {predicted_bytes_per_token/1024:.1f} KB')
print(f'  = 2 x {n_layers} layers x {n_kv_heads} kv-heads x {head_dim} head-dim x {elem_bytes} bytes')
print(f'\nAt {LONG_TOKENS} tokens of context, one user should need '
      f'{predicted_bytes_per_token*LONG_TOKENS/1e9:.2f} GB.')
if HAS_CUDA:
    print(f'Free VRAM / that figure -> about '
          f'{int(free_b / max(predicted_bytes_per_token*LONG_TOKENS, 1))} concurrent users at that context.')
print('\n(Prediction only. Measured in Section 6.)')

## 3. First Inference: Why Does the Long Prompt Feel Slower?

Two prompts, same model, same output length. One is a single sentence; the other is a
few thousand tokens. We measure **time to first token (TTFT)** — the pause the user
stares at — and total wall time.

In [ ]:
def build_prompt_ids(n_tokens):
    # Repeat filler text, then truncate to an exact token budget.
    reps = max(1, n_tokens // 8)
    enc = tokenizer(FILLER * reps, return_tensors='pt',
                    truncation=True, max_length=n_tokens)
    return enc['input_ids'].to(DEVICE)


def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


short_ids = tokenizer(SHORT_PROMPT, return_tensors='pt')['input_ids'].to(DEVICE)
long_ids = build_prompt_ids(LONG_TOKENS)

print(f'Short prompt : {short_ids.shape[1]:>5} tokens')
print(f'Long prompt  : {long_ids.shape[1]:>5} tokens')

In [ ]:
# Warm up on the LONGEST input first: the first call pays for CUDA kernel
# compilation and peak allocation. Measuring that would be measuring the wrong thing.
print('Warming up ...')
with torch.no_grad():
    model(input_ids=long_ids, use_cache=True)
sync()
if HAS_CUDA:
    torch.cuda.empty_cache()
print('Ready.')

In [ ]:
# TTFT and total latency for both prompts — a real generate() call, timed twice:
# once stopped after 1 token, once run to completion.
first_pass = []
for label, ids in (('short', short_ids), ('long', long_ids)):
    sync(); t0 = time.perf_counter()
    with torch.no_grad():
        model.generate(input_ids=ids, max_new_tokens=1, do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)
    sync(); ttft = time.perf_counter() - t0

    sync(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=DECODE_STEPS, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    sync(); total = time.perf_counter() - t0
    n_new = out.shape[1] - ids.shape[1]

    first_pass.append({'Prompt': label, 'Input tokens': ids.shape[1],
                       'Output tokens': n_new,
                       'TTFT (ms)': round(ttft * 1000, 1),
                       'Total (ms)': round(total * 1000, 1),
                       'TTFT share of total': f'{ttft/total*100:.0f}%'})
    print(f'{label:>5} prompt: TTFT {ttft*1000:7.1f} ms | total {total*1000:8.1f} ms')

first_table = pd.DataFrame(first_pass)
first_table

In [ ]:
fig, (ax_ttft, ax_split) = plt.subplots(1, 2, figsize=(12, 4))

labels = first_table['Prompt'] + '\n(' + first_table['Input tokens'].astype(str) + ' tok)'
ax_ttft.bar(labels, first_table['TTFT (ms)'], color=['#dcfce7', '#ffe4e6'],
            edgecolor='#000', linewidth=1.2)
ax_ttft.set_ylabel('TTFT (ms)', fontsize=11)
ax_ttft.set_title('Time to first token: the user-visible pause', fontsize=12, fontweight='bold')
for i, v in enumerate(first_table['TTFT (ms)']):
    ax_ttft.text(i, v + first_table['TTFT (ms)'].max() * 0.03, f'{v:.0f} ms',
                 ha='center', fontsize=11)

rest = first_table['Total (ms)'] - first_table['TTFT (ms)']
ax_split.bar(labels, first_table['TTFT (ms)'], color='#dbeafe', edgecolor='#000',
             linewidth=1.2, label='Waiting for token 1')
ax_split.bar(labels, rest, bottom=first_table['TTFT (ms)'], color='#fef3c7',
             edgecolor='#000', linewidth=1.2, label=f'Streaming the next {DECODE_STEPS-1}')
ax_split.set_ylabel('Wall time (ms)', fontsize=11)
ax_split.set_title('Same output length, very different shape', fontsize=12, fontweight='bold')
ax_split.legend(fontsize=9)

for ax in (ax_ttft, ax_split):
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

ratio = first_table.loc[1, 'TTFT (ms)'] / max(first_table.loc[0, 'TTFT (ms)'], 1e-9)
print(f'The long prompt waits {ratio:.1f}x longer for its first token.')
print('Same model, same output budget. The difference is all in the input.')

## 4. Splitting the Clock: Prefill vs Decode

`generate()` hides the two phases behind one number. So we stop using it and drive the
model by hand:

- **Prefill** — *one* forward pass over the whole prompt, `use_cache=True`. All prompt
  tokens go through the GPU in parallel. This produces the first token, so
  **prefill is TTFT**.
- **Decode** — one forward pass *per token*, feeding back a single token plus the
  `past_key_values` from the step before. Strictly sequential: token N needs token N-1.

Timing them separately is the whole point. Averaging them together is how teams end up
optimizing the wrong phase.

In [ ]:
# --- KV cache helpers -----------------------------------------------------
# transformers has changed its cache representation several times (legacy tuples,
# DynamicCache with key_cache/value_cache lists, and newer .layers objects).
# Normalize to a list of (key, value) tensors so the measurements below are
# version-independent.
def cache_layers(past):
    if past is None:
        return []
    if hasattr(past, 'to_legacy_cache'):
        try:
            legacy = past.to_legacy_cache()
            if legacy:
                return [(lay[0], lay[1]) for lay in legacy]
        except Exception:
            pass
    if hasattr(past, 'key_cache') and hasattr(past, 'value_cache'):
        return list(zip(past.key_cache, past.value_cache))
    if hasattr(past, 'layers'):
        try:
            return [(lay.keys, lay.values) for lay in past.layers]
        except Exception:
            pass
    if isinstance(past, (list, tuple)):
        return [(lay[0], lay[1]) for lay in past if lay is not None and len(lay) >= 2]
    return []


def cache_bytes(past):
    return sum(k.numel() * k.element_size() + v.numel() * v.element_size()
               for k, v in cache_layers(past))


def cache_seq_len(past):
    if past is None:
        return 0
    if hasattr(past, 'get_seq_length'):
        try:
            return int(past.get_seq_length())
        except Exception:
            pass
    lays = cache_layers(past)
    return int(lays[0][0].shape[-2]) if lays else 0


print(f'Cache helpers ready (transformers {__import__("transformers").__version__}).')

In [ ]:
def prefill_then_decode(prompt_ids, n_decode_steps):
    # PREFILL: one forward over the entire prompt, keeping the KV cache.
    sync(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model(input_ids=prompt_ids, use_cache=True)
    sync(); prefill_sec = time.perf_counter() - t0

    past = out.past_key_values
    next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    tokens = [int(next_tok.item())]

    # DECODE: one forward per token, 1 token in, cache reused.
    step_secs = []
    for _ in range(n_decode_steps):
        sync(); s0 = time.perf_counter()
        with torch.no_grad():
            out = model(input_ids=next_tok, past_key_values=past, use_cache=True)
        sync(); step_secs.append(time.perf_counter() - s0)

        past = out.past_key_values
        next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tokens.append(int(next_tok.item()))

    n_prompt = int(prompt_ids.shape[1])
    decode_sec = sum(step_secs)
    return {
        'prompt_tokens': n_prompt,
        'prefill_sec': prefill_sec,
        'decode_sec': decode_sec,
        'step_secs': step_secs,
        'prefill_tok_s': n_prompt / prefill_sec,
        'decode_tok_s': len(step_secs) / decode_sec if decode_sec else 0.0,
        'mean_itl_ms': (decode_sec / len(step_secs) * 1000) if step_secs else 0.0,
        'kv_bytes': cache_bytes(past),
        'kv_tokens': cache_seq_len(past),
        'text': tokenizer.decode(tokens, skip_special_tokens=True),
    }


runs = {}
for label, ids in (('short', short_ids), ('long', long_ids)):
    print(f'Running {label} prompt ({ids.shape[1]} tokens) ...')
    runs[label] = prefill_then_decode(ids, DECODE_STEPS)
    r = runs[label]
    print(f'  prefill {r["prefill_sec"]*1000:8.1f} ms  ({r["prefill_tok_s"]:8.0f} tok/s)')
    print(f'  decode  {r["decode_sec"]*1000:8.1f} ms  ({r["decode_tok_s"]:8.1f} tok/s, '
          f'{r["mean_itl_ms"]:.1f} ms/token)')
    print(f'  continuation: {r["text"][:70]!r}')

In [ ]:
phase_table = pd.DataFrame([{
    'Prompt': label,
    'Input tokens': r['prompt_tokens'],
    'Prefill (ms)': round(r['prefill_sec'] * 1000, 1),
    'Prefill (tok/s)': round(r['prefill_tok_s']),
    'Decode steps': len(r['step_secs']),
    'Mean ITL (ms/tok)': round(r['mean_itl_ms'], 2),
    'Decode (tok/s)': round(r['decode_tok_s'], 1),
    'Prefill:decode throughput': f"{r['prefill_tok_s']/max(r['decode_tok_s'], 1e-9):.0f}x",
} for label, r in runs.items()])
phase_table

In [ ]:
fig, (ax_share, ax_steps) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: how the wall clock divides between the two phases.
names = list(runs.keys())
pre_ms = [runs[k]['prefill_sec'] * 1000 for k in names]
dec_ms = [runs[k]['decode_sec'] * 1000 for k in names]
xlabels = [f"{k}\n({runs[k]['prompt_tokens']} tok in)" for k in names]

ax_share.bar(xlabels, pre_ms, color='#dbeafe', edgecolor='#000', linewidth=1.2,
             label='Prefill (parallel, 1 pass)')
ax_share.bar(xlabels, dec_ms, bottom=pre_ms, color='#ffe4e6', edgecolor='#000',
             linewidth=1.2, label=f'Decode ({DECODE_STEPS} passes, sequential)')
ax_share.set_ylabel('Wall time (ms)', fontsize=11)
ax_share.set_title('Where the time actually goes', fontsize=12, fontweight='bold')
ax_share.legend(fontsize=9)
for i, (p, d) in enumerate(zip(pre_ms, dec_ms)):
    ax_share.text(i, p + d + (max(pre_ms) + max(dec_ms)) * 0.02,
                  f'prefill {p/(p+d)*100:.0f}%', ha='center', fontsize=9)

# Right: per-step decode latency. Flat line = each token costs the same.
for k, color in zip(names, ('#2563eb', '#991b1b')):
    steps = [s * 1000 for s in runs[k]['step_secs']]
    ax_steps.plot(range(1, len(steps) + 1), steps, marker='o', markersize=3,
                  linewidth=1.5, color=color,
                  label=f"{k} prompt (mean {runs[k]['mean_itl_ms']:.1f} ms)")
ax_steps.set_xlabel('Decode step', fontsize=11)
ax_steps.set_ylabel('Time per token (ms)', fontsize=11)
ax_steps.set_title('Decode cost per token, step by step', fontsize=12, fontweight='bold')
ax_steps.set_ylim(bottom=0)
ax_steps.legend(fontsize=9)

for ax in (ax_share, ax_steps):
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

### Read the Two Panels Carefully

**Left:** for the short prompt, prefill is a rounding error and decode is the whole
bill. For the long prompt, prefill is suddenly a serious chunk of the wall clock. Same
model, opposite bottleneck — which is why one slow metric is never enough to tell you
what to fix.

**Right:** decode cost per token is roughly *flat*, and the long prompt's line sits only
slightly above the short one's. Decode does not care much how much context you already
have — it re-reads the model weights every single step regardless. That is the clue we
chase in Section 5.

The cell below makes the asymmetry explicit: how many tokens per second each phase
sustains on this exact GPU.

In [ ]:
print('--- Throughput by phase (same GPU, same model) ---')
for label, r in runs.items():
    print(f'{label:>5} prompt: prefill {r["prefill_tok_s"]:>9.0f} tok/s  |  '
          f'decode {r["decode_tok_s"]:>7.1f} tok/s  |  '
          f'ratio {r["prefill_tok_s"]/max(r["decode_tok_s"], 1e-9):>6.0f}x')

print('\nPrefill pushes orders of magnitude more tokens per second than decode.')
print('Prefill does one big parallel pass; decode does one tiny pass per token and')
print('pays the full cost of reading the weights each time.')

### Does Prefill Really Scale With Prompt Length?

One long prompt is an anecdote. Let's sweep the prompt length and measure both phases at
each point — prefill time should climb with input length, mean decode time should barely
move.

In [ ]:
sweep = []
for n_tok in SWEEP_LENGTHS:
    ids = build_prompt_ids(n_tok)
    r = prefill_then_decode(ids, max(8, DECODE_STEPS // 4))
    sweep.append({'Input tokens': r['prompt_tokens'],
                  'Prefill (ms)': round(r['prefill_sec'] * 1000, 1),
                  'Prefill (tok/s)': round(r['prefill_tok_s']),
                  'Mean ITL (ms/tok)': round(r['mean_itl_ms'], 2),
                  'KV cache (MB)': round(r['kv_bytes'] / 1e6, 1)})
    print(f'  {r["prompt_tokens"]:>5} tok -> prefill {r["prefill_sec"]*1000:8.1f} ms | '
          f'ITL {r["mean_itl_ms"]:6.2f} ms | KV {r["kv_bytes"]/1e6:7.1f} MB')
    del ids
    if HAS_CUDA:
        torch.cuda.empty_cache()

sweep_table = pd.DataFrame(sweep)
sweep_table

In [ ]:
fig, (ax_p, ax_d) = plt.subplots(1, 2, figsize=(13, 4.5))

ax_p.plot(sweep_table['Input tokens'], sweep_table['Prefill (ms)'],
          marker='o', color='#2563eb', linewidth=2)
ax_p.set_xlabel('Input tokens', fontsize=11)
ax_p.set_ylabel('Prefill time (ms)', fontsize=11)
ax_p.set_title('Prefill: grows with the prompt', fontsize=12, fontweight='bold')
ax_p.set_ylim(bottom=0)
for x, y in zip(sweep_table['Input tokens'], sweep_table['Prefill (ms)']):
    ax_p.annotate(f'{y:.0f}', (x, y), textcoords='offset points', xytext=(0, 7),
                  ha='center', fontsize=9)

ax_d.plot(sweep_table['Input tokens'], sweep_table['Mean ITL (ms/tok)'],
          marker='s', color='#991b1b', linewidth=2)
ax_d.set_xlabel('Input tokens', fontsize=11)
ax_d.set_ylabel('Mean time per decoded token (ms)', fontsize=11)
ax_d.set_title('Decode: nearly flat, whatever the context', fontsize=12, fontweight='bold')
ax_d.set_ylim(0, max(sweep_table['Mean ITL (ms/tok)']) * 1.6)

for ax in (ax_p, ax_d):
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

growth = sweep_table['Prefill (ms)'].iloc[-1] / max(sweep_table['Prefill (ms)'].iloc[0], 1e-9)
itl_growth = sweep_table['Mean ITL (ms/tok)'].iloc[-1] / max(sweep_table['Mean ITL (ms/tok)'].iloc[0], 1e-9)
print(f'{sweep_table["Input tokens"].iloc[0]} -> {sweep_table["Input tokens"].iloc[-1]} tokens:')
print(f'  prefill time   x{growth:.1f}')
print(f'  per-token decode time x{itl_growth:.2f}')
print('\nNote the y-axes: the left plot starts at 0 and climbs; the right one is')
print('deliberately zoomed out so a flat line looks flat instead of looking noisy.')

## 5. Why Can't Decode Go Faster?

Here is the claim: **every decode step has to read the entire model from memory** to
produce one token. If that is true, decode has a hard floor set by memory bandwidth, not
by how fast the GPU can multiply.

Rather than quoting a spec sheet, we measure this machine's achievable bandwidth with a
large device-to-device copy, then compare the floor it implies against the per-token
latency we measured above.

In [ ]:
def measure_bandwidth_gbs(iters=20):
    # Large fp16 copy. Each iteration reads nbytes and writes nbytes.
    if not HAS_CUDA:
        return None
    free_b_now, _ = torch.cuda.mem_get_info()
    nbytes = int(min(512e6, free_b_now * 0.15))
    n_elem = max(nbytes // 2, 1 << 20)
    try:
        src = torch.empty(n_elem, dtype=torch.float16, device='cuda')
        dst = torch.empty_like(src)
    except torch.cuda.OutOfMemoryError:
        return None

    for _ in range(3):
        dst.copy_(src)
    torch.cuda.synchronize()

    best = float('inf')
    for _ in range(iters):
        t0 = time.perf_counter()
        dst.copy_(src)
        torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)

    moved = 2 * src.numel() * src.element_size()  # read + write
    del src, dst
    torch.cuda.empty_cache()
    return moved / best / 1e9


bw_gbs = measure_bandwidth_gbs()
if bw_gbs:
    print(f'Measured achievable bandwidth: {bw_gbs:.0f} GB/s')
    print('(A copy benchmark — a practical floor, below the vendor peak number.)')
else:
    print('Bandwidth benchmark skipped (no CUDA or not enough free memory).')

In [ ]:
if bw_gbs:
    rows = []
    for label, r in runs.items():
        weights_ms = param_bytes / (bw_gbs * 1e9) * 1000
        kv_ms = r['kv_bytes'] / (bw_gbs * 1e9) * 1000
        floor_ms = weights_ms + kv_ms
        rows.append({
            'Prompt': label,
            'Weights read (GB)': round(param_bytes / 1e9, 2),
            'KV read (GB)': round(r['kv_bytes'] / 1e9, 3),
            'Floor from bandwidth (ms/tok)': round(floor_ms, 2),
            'Measured ITL (ms/tok)': round(r['mean_itl_ms'], 2),
            'Bandwidth efficiency': f'{floor_ms/r["mean_itl_ms"]*100:.0f}%',
        })
    floor_table = pd.DataFrame(rows)
    display(floor_table)

    print(f'\nReading {param_bytes/1e9:.1f} GB of weights at {bw_gbs:.0f} GB/s costs '
          f'{param_bytes/(bw_gbs*1e9)*1000:.1f} ms.')
    print('That happens once per token. It is the floor: no faster math removes it.')
    print('The gap between floor and measured is kernel launch overhead, attention,')
    print('sampling, and Python — all of it fixable. The floor is not.')
else:
    floor_table = None
    print('Skipped: needs a CUDA device.')

In [ ]:
if bw_gbs:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    names = floor_table['Prompt']
    x = range(len(names))
    width = 0.38

    ax.bar([i - width/2 for i in x], floor_table['Floor from bandwidth (ms/tok)'],
           width, color='#dbeafe', edgecolor='#000', linewidth=1.2,
           label='Floor: time to read weights + KV once')
    ax.bar([i + width/2 for i in x], floor_table['Measured ITL (ms/tok)'],
           width, color='#ffe4e6', edgecolor='#000', linewidth=1.2,
           label='Measured time per token')

    for i, (f, m) in enumerate(zip(floor_table['Floor from bandwidth (ms/tok)'],
                                   floor_table['Measured ITL (ms/tok)'])):
        ax.text(i - width/2, f + m * 0.03, f'{f:.1f}', ha='center', fontsize=10)
        ax.text(i + width/2, m + m * 0.03, f'{m:.1f}', ha='center', fontsize=10)

    ax.set_xticks(list(x))
    ax.set_xticklabels([f'{n} prompt' for n in names])
    ax.set_ylabel('ms per decoded token', fontsize=11)
    ax.set_title(f'Decode is bandwidth-bound: {bw_gbs:.0f} GB/s sets the floor',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()

## 6. The KV Cache: What Is Holding Decode Up

Decode reuses the keys and values computed for every earlier token. Without that cache,
each new token would have to re-run attention over the whole sequence from scratch.

Two things to settle with measurements:

1. **How much does the cache buy us?** Benchmark decode with `past_key_values` against
   decode with `use_cache=False`, where every step re-forwards the full sequence.
2. **Was the Section 2 prediction right?** We predicted bytes-per-token from the config.
   Now we can weigh the real cache.

In [ ]:
def decode_without_cache(prompt_ids, steps):
    # Every step re-forwards the entire sequence. This is the thing the cache removes.
    seq = prompt_ids.clone()
    with torch.no_grad():
        out = model(input_ids=seq, use_cache=False)
    next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    step_secs = []
    for _ in range(steps):
        seq = torch.cat([seq, next_tok], dim=1)
        sync(); t0 = time.perf_counter()
        with torch.no_grad():
            out = model(input_ids=seq, use_cache=False)
        sync(); step_secs.append(time.perf_counter() - t0)
        next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    return step_secs


kv_rows, kv_curves = [], {}
for label, ids in (('short', short_ids), ('long', long_ids)):
    print(f'{label} prompt: {NO_CACHE_STEPS} steps with cache, then without ...')
    with_cache = prefill_then_decode(ids, NO_CACHE_STEPS)
    without_secs = decode_without_cache(ids, NO_CACHE_STEPS)

    with_ms = with_cache['mean_itl_ms']
    without_ms = sum(without_secs) / len(without_secs) * 1000
    measured_bpt = with_cache['kv_bytes'] / max(with_cache['kv_tokens'], 1)

    kv_curves[label] = {
        'with': [s * 1000 for s in with_cache['step_secs']],
        'without': [s * 1000 for s in without_secs],
    }
    kv_rows.append({
        'Prompt': label,
        'Input tokens': with_cache['prompt_tokens'],
        'With cache (ms/tok)': round(with_ms, 2),
        'Without cache (ms/tok)': round(without_ms, 2),
        'Speedup': f'{without_ms/with_ms:.1f}x',
        'Cached tokens': with_cache['kv_tokens'],
        'KV cache (MB)': round(with_cache['kv_bytes'] / 1e6, 2),
        'Measured B/token': round(measured_bpt),
        'Predicted B/token': predicted_bytes_per_token,
    })
    print(f'  with cache {with_ms:7.2f} ms/tok | without {without_ms:8.2f} ms/tok | '
          f'{without_ms/with_ms:.1f}x')
    if HAS_CUDA:
        torch.cuda.empty_cache()

kv_table = pd.DataFrame(kv_rows)
kv_table

In [ ]:
fig, (ax_cmp, ax_pred) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: with vs without cache, per step. The no-cache line should climb —
# each step re-forwards a longer sequence.
for label, color in zip(kv_curves, ('#2563eb', '#991b1b')):
    steps = range(1, len(kv_curves[label]['without']) + 1)
    ax_cmp.plot(steps, kv_curves[label]['without'], marker='o', markersize=3,
                linewidth=1.8, color=color, label=f'{label}: no cache')
    ax_cmp.plot(steps, kv_curves[label]['with'], marker='s', markersize=3,
                linewidth=1.8, color=color, linestyle='--', label=f'{label}: with cache')
ax_cmp.set_xlabel('Decode step', fontsize=11)
ax_cmp.set_ylabel('Time per token (ms)', fontsize=11)
ax_cmp.set_title('The KV cache turns O(n) work per token into O(1)',
                 fontsize=12, fontweight='bold')
ax_cmp.set_yscale('log')
ax_cmp.legend(fontsize=8)

# Right: did the config-derived prediction hold up?
x = range(len(kv_table))
width = 0.38
ax_pred.bar([i - width/2 for i in x], kv_table['Predicted B/token'] / 1024, width,
            color='#dbeafe', edgecolor='#000', linewidth=1.2, label='Predicted (config)')
ax_pred.bar([i + width/2 for i in x], kv_table['Measured B/token'] / 1024, width,
            color='#dcfce7', edgecolor='#000', linewidth=1.2, label='Measured (real cache)')
ax_pred.set_xticks(list(x))
ax_pred.set_xticklabels([f'{p} prompt' for p in kv_table['Prompt']])
ax_pred.set_ylabel('KB per cached token', fontsize=11)
ax_pred.set_title('Prediction vs measurement', fontsize=12, fontweight='bold')
ax_pred.legend(fontsize=9)
for i, (p, m) in enumerate(zip(kv_table['Predicted B/token'], kv_table['Measured B/token'])):
    ax_pred.text(i - width/2, p/1024 * 1.02, f'{p/1024:.0f}', ha='center', fontsize=10)
    ax_pred.text(i + width/2, m/1024 * 1.02, f'{m/1024:.0f}', ha='center', fontsize=10)

for ax in (ax_cmp, ax_pred):
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

err = abs(kv_table['Measured B/token'].iloc[0] - predicted_bytes_per_token) / predicted_bytes_per_token
print(f'Prediction error: {err*100:.1f}%')
print('If this is near zero, the memory equation from Section 2 is trustworthy —')
print('you can size a deployment from config.json before renting a single GPU.')
print('If it is far off, suspect a non-standard cache layout (sliding window,')
print('MLA, or a quantized cache) and go read the model code.')

## 7. What We Measured

| Finding | Evidence in this notebook |
|---------|---------------------------|
| Weights are a fixed cost; the KV cache is the variable one | §2 memory accounting, §6 bytes/token |
| TTFT is set by prompt length | §3 short vs long, §4 prefill sweep |
| Prefill and decode are different workloads with different bottlenecks | §4 phase split and throughput ratio |
| Decode cost per token is flat in context, and orders of magnitude slower than prefill | §4 right-hand panel |
| Decode is bandwidth-bound — the floor is "read the whole model, once per token" | §5 measured bandwidth vs measured ITL |
| The KV cache is what makes decode O(1) instead of O(n) per token | §6 with vs without |

### The two numbers to take away

1. **Prefill tokens/sec ÷ decode tokens/sec** on your hardware. That ratio is why
   batching decode is nearly free and why one giant prompt can stall a whole server.
2. **Weight bytes ÷ measured bandwidth.** That is your per-token floor. Every serving
   optimization worth knowing — quantization, GQA/MLA, speculative decoding,
   batching — is an attack on either the numerator or how many tokens you get per read.

### Where this goes next

- **Quantization** (`demo_e_quantization_batching.ipynb`) — shrink the numerator.
- **Attention variants** — GQA and MLA shrink the KV cache instead.
- **Batching and engines** — amortize one weight read across many users' tokens.

> **Caveat for the honest reader:** these numbers describe *this* GPU, *this* model, and
> an unbatched, pure-PyTorch loop. A real engine (vLLM, SGLang, TensorRT-LLM) will beat
> every absolute figure here. None of that changes the shape of the curves — which is
> what the rest of the workshop is about.